In [1]:
import os
import re
import time
import warnings

from google.adk.agents.llm_agent import LlmAgent
from google.adk.runners import Runner
from google.adk.sessions.in_memory_session_service import InMemorySessionService
from google.genai import types
from IPython.display import clear_output, display as ipy_display
import ipywidgets as widgets
import requests
import vertexai

# Suppress experimental JSON schema warning
warnings.filterwarnings(
    "ignore",
    message=".*JSON_SCHEMA_FOR_FUNC_DECL is enabled.*"
)

# ------------------------------------------------------------
# 1. INITIALIZATION & SETUP
# ------------------------------------------------------------

PROJECT_ID = ! gcloud config get-value project
PROJECT_ID = PROJECT_ID[0]
LOCATION = "us-central1"  # @param {type:"string"}

# define project information manually if the above code didn't work
if PROJECT_ID == "(unset)":
    PROJECT_ID = "qwiklabs-gcp-02-c413994bc4ba"  # @param {type:"string"}

print(f"Using Project ID: {PROJECT_ID}")

os.environ["GOOGLE_CLOUD_PROJECT"] = PROJECT_ID
os.environ["GOOGLE_CLOUD_QUOTA_PROJECT"] = PROJECT_ID
os.environ["ADK_SUPPRESS_GEMINI_LITELLM_WARNINGS"] = "true"

vertexai.init(project=PROJECT_ID, location=LOCATION)

session_service = InMemorySessionService()

# Lists to keep logs programmatically as requested
prompt_logs = []
response_logs = []

# ------------------------------------------------------------
# CALLBACKS FOR LOGGING & VALIDATION
# ------------------------------------------------------------


def validate_user_input(session, message):
    """Callback to check and validate user input before sending to model."""
    if not message or not message.parts:
        raise ValueError("Empty message encountered.")

    user_text = " ".join(
        [part.text for part in message.parts if part.text]
    ).strip()

    if len(user_text) < 2:
        raise ValueError("Query is too short. Please provide a location.")

    # Security / Anti-Injection Validation
    malicious_patterns = [
        r"(?i)system\s+prompt",
        r"(?i)ignore\s+previous",
        r"(?i)delete\s+from",
        r"(?i)drop\s+table",
        r"(?i)select\s+\*\s+from"
    ]
    for pattern in malicious_patterns:
        if re.search(pattern, user_text):
            raise ValueError(
                "Security violation: Input contains prohibited commands "
                "or prompt injection."
            )

    # US Location Check Validation
    non_us_indicators = [
        "london", "paris", "tokyo", "berlin", "beijing", "sydney", "rome",
        "canada", "mexico", "france", "germany", "uk", "united kingdom",
        "japan", "australia"
    ]
    normalized_text = user_text.lower()
    for indicator in non_us_indicators:
        if f", {indicator}" in normalized_text or normalized_text == indicator:
            raise ValueError(
                f"NWS Forecast only supports United States locations. "
                f"'{user_text}' is recognized as outside the US."
            )

    return message


def log_user_prompt(session, message):
    """Callback to log user prompts."""
    global prompt_logs
    user_text = " ".join(
        [part.text for part in message.parts if part.text]
    ).strip()
    log_entry = f"[Session: {session.session_id}] Prompt: {user_text}"
    prompt_logs.append(log_entry)
    print(f"[CALLBACK LOG - USER PROMPT]: {user_text}")


def log_model_response(session, message):
    """Callback to log model responses."""
    global response_logs
    model_text = " ".join(
        [part.text for part in message.parts if part.text]
    ).strip()
    log_entry = f"[Session: {session.session_id}] Response: {model_text}"
    response_logs.append(log_entry)
    print(f"[CALLBACK LOG - MODEL RESPONSE]: {model_text}")

# ------------------------------------------------------------
# 2. DEFINE THE TWO AGENTS & RUNNERS (FLASH & PRO)
# ------------------------------------------------------------


agent_instruction = (
    "You are a helpful weather assistant. "
    "When a user provides a location, use your geocoding and forecast tools "
    "to find the weather forecast for that location. "
    "If the location is generic (like 'Chicago' or 'Lancaster'), proceed with "
    "geocoding the most probable option. "
    "IMPORTANT: Always explicitly state the identified state (or region) "
    "and country for the city you chose in your final response to the user, "
    "so they know exactly which location's forecast you retrieved (e.g., "
    "'Retrieving weather for Lancaster, Pennsylvania, United States'). "
    "If the user clarifies a specific state or country, use that precise "
    "location for geocoding."
)

# Flash Agent and Runner
agent_flash = LlmAgent(
    name="VertexFlashWeather",
    model="vertex_ai/gemini-2.5-flash",
    instruction=agent_instruction,
)

runner_flash = Runner(
    agent=agent_flash,
    app_name="VertexFlashApp",
    session_service=session_service,
    auto_create_session=True,
)

# Pro Agent and Runner
agent_pro = LlmAgent(
    name="VertexProWeather",
    model="vertex_ai/gemini-2.5-pro",
    instruction=agent_instruction,
)

runner_pro = Runner(
    agent=agent_pro,
    app_name="VertexProApp",
    session_service=session_service,
    auto_create_session=True,
)

# Assign Callback Lists directly to Runner Attributes
runner_flash.pre_run_callbacks = [validate_user_input, log_user_prompt]
runner_flash.post_run_callbacks = [log_model_response]

runner_pro.pre_run_callbacks = [validate_user_input, log_user_prompt]
runner_pro.post_run_callbacks = [log_model_response]


# Assign tools
def geocode_place_free(address: str) -> str:
    """Geocode an address into latitude and longitude coordinates."""
    base_url = "https://nominatim.openstreetmap.org/search"
    params = {"q": address, "format": "json", "limit": 1}
    headers = {
        "User-Agent": "RussellWeatherApp/1.0 (russell.f.zimmermann@saic.com)"
    }
    try:
        response = requests.get(
            base_url, params=params, headers=headers, timeout=10
        )
        response.raise_for_status()
        data = response.json()
        if data:
            return f"{data[0]['lat']}, {data[0]['lon']}"
    except Exception:
        pass
    return "Error: Location not found."


def get_nws_forecast(latitude: float, longitude: float) -> str:
    """Retrieve the National Weather Service (NWS) forecast."""
    headers = {
        "User-Agent": "RussellWeatherApp/1.0 (russell.f.zimmermann@saic.com)"
    }
    points_url = f"https://api.weather.gov/points/{latitude},{longitude}"
    try:
        points_response = requests.get(
            points_url, headers=headers, timeout=10
        )
        points_response.raise_for_status()
        forecast_url = points_response.json()["properties"]["forecast"]
        forecast_response = requests.get(
            forecast_url, headers=headers, timeout=10
        )
        forecast_response.raise_for_status()
        periods = forecast_response.json()["properties"]["periods"]
        formatted = []
        for period in periods[:2]:
            formatted.append(
                f"{period['name']}: {period['temperature']}°"
                f"{period['temperatureUnit']}, {period['shortForecast']}."
            )
        return "\n\n".join(formatted)
    except Exception as e:
        return f"Error: {e}"


agent_flash.tools = [geocode_place_free, get_nws_forecast]
agent_pro.tools = [geocode_place_free, get_nws_forecast]

# ------------------------------------------------------------
# 4. SIMPLIFIED INTERACTIVE CHAT INTERFACE
# ------------------------------------------------------------

session_id = "weather_session"
user_id = "russell"

chat_output = widgets.Output()

model_dropdown = widgets.Dropdown(
    options=[("Gemini 2.5 Flash", "flash"), ("Gemini 2.5 Pro", "pro")],
    value="flash",
    description="Select Model",
    style={"description_width": "110px"},
    layout=widgets.Layout(width="260px"),
)

custom_css = """
<style>
.widget-dropdown select {
    background-color: #212529 !important;
    color: #ffffff !important;
    border: 1px solid #495057 !important;
}
</style>
"""
css_widget = widgets.HTML(custom_css)

user_text_input = widgets.Text(
    placeholder="Type your location or reply here and press Enter...",
    layout=widgets.Layout(width="60%"),
)

send_button = widgets.Button(
    description="Send",
    button_style="info",
    layout=widgets.Layout(width="15%")
)
exit_button = widgets.Button(
    description="End Chat",
    button_style="danger",
    layout=widgets.Layout(width="15%")
)

top_row = widgets.HBox(
    [model_dropdown],
    layout=widgets.Layout(grid_gap="20px")
)
bottom_row = widgets.HBox(
    [user_text_input, send_button, exit_button],
    layout=widgets.Layout(grid_gap="10px")
)
controls = widgets.VBox(
    [top_row, bottom_row],
    layout=widgets.Layout(margin="10px 0px")
)

interface_box = widgets.VBox([chat_output, controls])


def append_message(sender, text, is_agent=False):
    with chat_output:
        color = "#0d6efd" if not is_agent else "#198754"
        sender_name = "You" if not is_agent else "Agent"
        ipy_display(widgets.HTML(
            f"<b style='color: {color};'>{sender_name}:</b> {text}<br>"
        ))


def disable_interface():
    model_dropdown.disabled = True
    user_text_input.disabled = True
    send_button.disabled = True
    exit_button.disabled = True
    with chat_output:
        ipy_display(widgets.HTML(
            "<hr><b style='color: #dc3545;'>Conversation ended.</b>"
        ))


def handle_submit(sender):
    user_query = user_text_input.value.strip()
    if not user_query:
        return
    user_text_input.value = ""

    append_message(user_id, user_query, is_agent=False)
    if user_query.lower() in ["exit", "quit"]:
        disable_interface()
        return

    selected_model_key = model_dropdown.value
    active_runner = runner_pro if selected_model_key == "pro" else runner_flash
    model_display_name = (
        "Gemini 2.5 Pro" if selected_model_key == "pro"
        else "Gemini 2.5 Flash"
    )

    with chat_output:
        ipy_display(widgets.HTML(
            f"<small style='color: #6c757d; font-style: italic;'>"
            f"[Routing message to {model_display_name}]</small>"
        ))

    message_content = types.Content(
        parts=[types.Part.from_text(text=user_query)]
    )
    try:
        events = active_runner.run(
            user_id=user_id,
            session_id=session_id,
            new_message=message_content
        )
        for event in events:
            if event.content and event.content.parts:
                for part in event.content.parts:
                    if part.text:
                        append_message("agent", part.text, is_agent=True)
    except Exception as e:
        append_message("system", f"Error validation: {e}", is_agent=True)


user_text_input.on_submit(handle_submit)
send_button.on_click(handle_submit)
exit_button.on_click(lambda b: disable_interface())

clear_output()
ipy_display(css_widget)
ipy_display(interface_box)

with chat_output:
    ipy_display(widgets.HTML(
        "<b style='color: #6c757d;'>Weather Agent initialized. "
        "Choose your model and type a location below:</b><br>"
    ))

HTML(value='\n<style>\n.widget-dropdown select {\n    background-color: #212529 !important;\n    color: #fffff…